# 7.1 LangChain基础模块

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
import os
os.environ["OPENAI_API_BASE"]  = "xxx"
os.environ["OPENAI_API_KEY"] = "xxx"
chat = ChatOpenAI(
    model="deepseek-r1:1.5b",
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)
messages = [
    SystemMessage(content="You are a helpful assistant that translates English to Chinese."),
    HumanMessage(content="I love programming."),
]
chat.invoke(messages)

AIMessage(content='\n\n我热爱编程！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 18, 'total_tokens': 27, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'deepseek-r1:1.5b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-106', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07d0b-51aa-7093-9ab4-df57c577d488-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 9, 'total_tokens': 27, 'input_token_details': {}, 'output_token_details': {}})

In [10]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./test.txt")
print(loader.load())

[Document(metadata={'source': './test.txt'}, page_content='Today is Sunday.\nI am a boy.\n')]


In [8]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.docstore.document import Document
import os

os.environ["OPENAI_API_BASE"]  = "xxx"
os.environ["OPENAI_API_KEY"] = "xxx"

raw_documents = [Document(page_content="葡萄", metadata={"source": "local"}),
Document(page_content="白菜", metadata={"source": "local"}),
Document(page_content="狗", metadata={"source": "local"})]
db = Chroma.from_documents(raw_documents, OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
))
query = "动物"
docs = db.similarity_search(query)
print(docs[0].page_content)

狗


# 7.2.基于LangChain实现RAG

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.schema.runnable import RunnablePassthrough
from langchain_classic.schema.output_parser import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage
import os

os.environ["OPENAI_API_BASE"]  = "xxx"
os.environ["OPENAI_API_KEY"] = "xxx"

loader = TextLoader('7.2_RAG_demo.txt',encoding = "utf-8")
documents = loader.load()
text_splitter = \
RecursiveCharacterTextSplitter(
chunk_size=100, chunk_overlap=80,separators=["\n\n", "\n", "。", "！", "？", "；", " ", ""])
chunks = text_splitter.split_documents(documents)
template = """你是一位问答助手,你的任务是根据###中间的文本信息回答问题，请准确回答问题，不要健谈，如果提供的文本信息无法回答问题，请直接回复“提供的文本无法回答问题”，我相信你能做的很好。###\n{context}###\n问题：{question}"""
question = "战士金喜欢哪写乐队？"
db = Chroma.from_documents(chunks, OllamaEmbeddings(model="bge-m3"))
retriever = db.as_retriever(search_kwargs={"k": 3})
context =  retriever.invoke(question)
print(context)

C:\Users\HGZ\AppData\Local\Temp\ipykernel_34384\2972321179.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '7.2_RAG_demo.txt'}, page_content='谷清水的昵称是战士金\n三位作者均在互联网行业工作\n三位作者均为男性\n谷清水是万能青年旅店、腰、华北浪革、生祥、声音碎片等乐队的粉丝\n三位作者在不同的城市'), Document(metadata={'source': '7.2_RAG_demo.txt'}, page_content='《大模型RAG实战》的作者是汪鹏、谷清水和卞龙鹏\n谷清水的昵称是战士金\n三位作者均在互联网行业工作\n三位作者均为男性\n谷清水是万能青年旅店、腰、华北浪革、生祥、声音碎片等乐队的粉丝')]


In [28]:
context_str = "；".join([d.page_content for d in context])
input_str = template.format_map({"context":context_str,"question":question})
chat = ChatOpenAI(
    model="deepseek-r1:7b",
    base_url="http://localhost:11434/v1",
    max_tokens=2048,
    temperature=0

)
messages = [SystemMessage(content="你是一位问答助手"), HumanMessage(content=input_str)]
response = chat.invoke(messages)
print(response)



战士金喜欢的乐队包括万能青年旅店、腰、华北浪革、生祥和声音碎片。
